# 🌲 Multi‑Class Covertype Classification
### With SHAP & LIME Explainability
**Created by Harivansh Bhardwaj**
---

In [ ]:
!pip install ucimlrepo shap lime

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import shap
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
covertype = fetch_ucirepo(id=31)
X = pd.DataFrame(covertype.data.features)
y = covertype.data.targets
print(X.shape, y.value_counts().head())

In [ ]:
# Remove highly correlated features instead of VIF (faster)
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X = X.drop(columns=to_drop)
print('Dropped due to correlation:', to_drop)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
# SHAP TreeExplainer (fast)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:200])
shap.summary_plot(shap_values, X_test[:200], feature_names=X.columns)

In [ ]:
lime_exp = LimeTabularExplainer(X_train[:200], feature_names=X.columns, class_names=np.unique(y).astype(str), discretize_continuous=True)
exp = lime_exp.explain_instance(X_test[0], model.predict_proba)
exp.show_in_notebook()